# 10 — Fine-tuning: Zoobot ConvNeXt-Nano (Local — RTX 5060 Ti 16 GB)

Fine-tuning de **Zoobot (ConvNeXt-Nano)** preentrenado con **92M+ anotaciones de Galaxy Zoo** para clasificación morfológica de galaxias (6 clases).  
Versión local optimizada para **NVIDIA RTX 5060 Ti 16 GB** (Blackwell, sm_120).

| Hiperparámetro | Valor |
|---|---|
| Batch size | 32 (16 GB VRAM) — reducir a 16 en 8 GB |
| LR backbone | 1e-5 |
| LR head | 1e-3 |
| Optimizer | AdamW (wd=0.01) |
| Scheduler | CosineAnnealingLR |
| Epochs | 50 (+ early stopping, paciencia=10) |
| AMP | ✅ float16 |
| torch.compile | ✅ Linux / ❌ Windows (sin Triton) |
| Early stopping | ✅ paciencia=10 epochs sin mejora en val F1 |
| IMAGE_SIZE | 224 px (resolución nativa Zoobot) |
| Layer Decay | 0.75 (regularización diferencial por profundidad) |
| Fine-tuning Mode | Full (backbone + head) |

> **¿Por qué Zoobot?**  
> Zoobot es un framework especializado en clasificación de morfología de galaxias, entrenado con >92 millones de respuestas de voluntarios en Galaxy Zoo (DECaLS, GZ2, Hubble, CANDELS, etc.).  
> A diferencia de modelos preentrenados en ImageNet (gatos, perros, autos), **Zoobot ya entiende la estructura de galaxias**: brazos espirales, barras, bordes, formas elípticas e irregulares.  
> Esto le da una **ventaja masiva** para fine-tuning en clasificación morfológica.

> **¿Por qué ConvNeXt-Nano?**  
> ConvNeXt-Nano es la arquitectura más ligera de Zoobot (~15M params), ideal para GPU de 16 GB.  
> Alternativas más grandes (convnext_small, maxvit_small) requieren más VRAM pero pueden dar mejor precisión.

> **Modo Full Fine-tuning:**  
> Entrenamos **todas las capas** (backbone + cabeza), no solo la cabeza.  
> Usamos **layer decay** (lr_decay=0.75) para que las capas más profundas aprendan más despacio, preservando las features astronómicas aprendidas por Zoobot.

> **Requisito de CUDA:** RTX 5060 Ti requiere **CUDA 12.8+** y **PyTorch ≥ 2.7**. Ejecuta primero la celda de instalación.

## Sección 0 — Instalación de dependencias

Ejecuta esta celda **una sola vez** en el entorno nuevo. Reinicia el kernel después si es la primera instalación.

**Zoobot** requiere: `pytorch-lightning>=2.1`, `timm>=0.9.2`, `albumentations`, `galaxy-datasets`.

In [ ]:
import subprocess, sys

# PyTorch con CUDA 12.8 (necesario para RTX 5060 Ti / Blackwell sm_120)
# Si ya tienes PyTorch >= 2.7 con CUDA 12.8, puedes saltar esta parte.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu128',
    '--quiet',
], check=True)

# Instalar Zoobot con soporte PyTorch
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--quiet',
    'zoobot[pytorch]',
], check=True)

# Dependencias adicionales
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--quiet',
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'Pillow', 'scikit-learn', 'tqdm',
], check=True)

print('Instalación completada. Reinicia el kernel si es la primera vez.')

## Sección 1 — Imports

In [ ]:
import gc
import os
import sys
import time
import pathlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from PIL import Image
from tqdm.auto import tqdm

# Zoobot imports
from zoobot.pytorch.training import finetune

warnings.filterwarnings('ignore')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    cap = torch.cuda.get_device_capability(0)
    print(f'Compute  : sm_{cap[0]}{cap[1]}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')

try:
    import zoobot
    print(f'Zoobot   : {zoobot.__version__}')
except AttributeError:
    print('Zoobot   : instalado (versión no disponible via __version__)')

## Sección 2 — Configuración

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Paths locales (relativo a la raíz del repo)
# ──────────────────────────────────────────────────────────────────────────────
_LOCAL     = pathlib.Path('../data')
IMAGES_DIR = _LOCAL / 'images_gz2' / 'images'
SPLITS_DIR = _LOCAL / 'splits'
CKPT_DIR   = pathlib.Path('../models/checkpoints/zoobot_convnext_nano')
LOG_DIR    = pathlib.Path('../logs')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Verificar que los datos existen
assert IMAGES_DIR.exists(), f'No se encontró IMAGES_DIR: {IMAGES_DIR.resolve()}'
assert (SPLITS_DIR / 'train.csv').exists(), f'No se encontró train.csv en {SPLITS_DIR.resolve()}'

# ──────────────────────────────────────────────────────────────────────────────
# Modelo Zoobot
# ──────────────────────────────────────────────────────────────────────────────
MODEL_NAME   = 'zoobot_convnext_large'
# Encoder preentrenado desde HuggingFace Hub
# Alternativas:
#   'hf_hub:mwalmsley/zoobot-encoder-convnext_small'  (más params, mejor acc)
#   'hf_hub:mwalmsley/zoobot-encoder-convnext_large'  (máxima acc, más VRAM)
#   'hf_hub:mwalmsley/zoobot-encoder-efficientnet_v2_s'
#   'hf_hub:mwalmsley/zoobot-encoder-maxvit_small'
ENCODER_NAME = 'hf_hub:mwalmsley/zoobot-encoder-convnext_large'

NUM_CLASSES  = 6
CLASS_ORDER  = ['Elliptical', 'Lenticular', 'Spiral', 'Barred_Spiral', 'Edge_on', 'Irregular']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# ──────────────────────────────────────────────────────────────────────────────
# Training — Full fine-tuning con layer decay
# ──────────────────────────────────────────────────────────────────────────────
EPOCHS         = 50       # más epochs que otros modelos para explorar el pico
EARLY_STOP_PAT = 10      # paciencia mayor: Zoobot converge más lento en fine-tuning completo
BATCH_SIZE     = 32      # seguro para 16 GB VRAM con AMP

NUM_WORKERS    = 0 if os.name == 'nt' else 4

# LR diferencial: backbone bajo (preservar features astronómicas), head alto
LR_BACKBONE  = 1e-5      # más bajo que ImageNet models — Zoobot ya sabe de galaxias
LR_HEAD      = 1e-3
WEIGHT_DECAY = 0.01      # mayor que otros modelos para regularización
LAYER_DECAY  = 0.75      # cada capa más profunda recibe lr * 0.75
USE_AMP      = device.type == 'cuda'
USE_COMPILE  = os.name != 'nt'

# Zoobot usa imágenes de 224×224 como resolución nativa
IMAGE_SIZE    = 224
CROP_SIZE     = 320      # CenterCrop para eliminar bordes negros de GZ2 (424×424)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
RANDOM_SEED   = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'OS             : {"Windows" if os.name == "nt" else "Linux/macOS"}')
print(f'IMAGES_DIR     : {IMAGES_DIR.resolve()}')
print(f'SPLITS_DIR     : {SPLITS_DIR.resolve()}')
print(f'CKPT_DIR       : {CKPT_DIR.resolve()}')
print(f'MODEL          : {MODEL_NAME}')
print(f'ENCODER        : {ENCODER_NAME}')
print(f'EPOCHS         : {EPOCHS}  (early stop paciencia={EARLY_STOP_PAT})')
print(f'BATCH_SIZE     : {BATCH_SIZE}')
print(f'NUM_WORKERS    : {NUM_WORKERS}')
print(f'LR backbone    : {LR_BACKBONE}')
print(f'LR head        : {LR_HEAD}')
print(f'WEIGHT_DECAY   : {WEIGHT_DECAY}')
print(f'LAYER_DECAY    : {LAYER_DECAY}')
print(f'USE_AMP        : {USE_AMP}')
print(f'USE_COMPILE    : {USE_COMPILE}')

## Sección 3 — Pipeline de datos

In [ ]:
# Transforms
# Zoobot fue preentrenado con normalización ImageNet y resolución 224.
# Para galaxias, la rotación es especialmente importante (sin orientación preferida).
train_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# Dataset
# Se almacenan numpy arrays en lugar de DataFrames para evitar copy-on-read
# en los worker processes (previene OOM de RAM).
class GalaxyDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform, class_to_idx):
        df = pd.read_csv(
            csv_path,
            usecols=['img_filename', 'morph_label'],
            dtype={'img_filename': 'str', 'morph_label': 'str'},
        )
        self.filenames = df['img_filename'].to_numpy()
        self.labels    = np.array(
            [class_to_idx[lbl] for lbl in df['morph_label']], dtype=np.int64
        )
        del df
        gc.collect()

        self.images_dir = pathlib.Path(images_dir)
        self.transform  = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        image = Image.open(self.images_dir / self.filenames[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])


# DataLoaders
g = torch.Generator().manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'train.csv', IMAGES_DIR, train_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'), generator=g,
)
val_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'val.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)

print(f'Train batches : {len(train_loader):,}  ({len(train_loader.dataset):,} imgs)')
print(f'Val   batches : {len(val_loader):,}  ({len(val_loader.dataset):,} imgs)')

In [ ]:
# Class weights para CrossEntropyLoss
_df_w = pd.read_csv(
    SPLITS_DIR / 'train.csv',
    usecols=['morph_label'],
    dtype={'morph_label': 'str'},
)
label_counts = np.bincount(
    _df_w['morph_label'].map(CLASS_TO_IDX).values,
    minlength=NUM_CLASSES,
)
weights      = len(_df_w) / (NUM_CLASSES * label_counts)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print('Class weights:')
for cls, w, n in zip(CLASS_ORDER, weights, label_counts):
    print(f'  {cls:<15}  n={n:>6,}   w={w:.4f}')

del _df_w
gc.collect()
print('RAM liberada ✓')

## Sección 4 — Modelo Zoobot

Usamos `FinetuneableZoobotClassifier` de Zoobot para crear el modelo.  
Internamente, esto carga el **encoder ConvNeXt-Nano** preentrenado con >92M anotaciones de Galaxy Zoo y le agrega una cabeza de clasificación para nuestras 6 clases.

Luego **extraemos el modelo PyTorch puro** para usarlo con nuestro pipeline manual de entrenamiento, manteniendo compatibilidad total con nuestros otros notebooks.

In [ ]:
# Crear el modelo Zoobot con FinetuneableZoobotClassifier
# Esto descarga automáticamente los pesos preentrenados desde HuggingFace Hub.
#
# n_blocks=0: NO congelar ningún bloque → full fine-tuning
# El modelo se usará con nuestro pipeline manual (no PyTorch Lightning)
zoobot_model = finetune.FinetuneableZoobotClassifier(
    name=ENCODER_NAME,
    num_classes=NUM_CLASSES,
    n_blocks=0,              # full fine-tune: 0 bloques congelados
    learning_rate=LR_BACKBONE,
    lr_decay=LAYER_DECAY,
)

# Extraer el modelo PyTorch puro (nn.Module) para nuestro pipeline manual.
# FinetuneableZoobotClassifier es un LightningModule que envuelve:
#   - self.encoder: el backbone preentrenado (ConvNeXt-Nano)
#   - self.head: la cabeza de clasificación (MLP)
# Podemos usar el modelo completo directamente como nn.Module.
model = zoobot_model

# Separar parámetros de backbone y cabeza para LR diferencial
backbone_params = []
head_params     = []

for name, param in model.named_parameters():
    if 'head' in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

print(f'Encoder: {ENCODER_NAME}')
print(f'Backbone params : {sum(p.numel() for p in backbone_params):,}')
print(f'Head params     : {sum(p.numel() for p in head_params):,}')

model = model.to(device)

# torch.compile() mejora el throughput en Linux mediante fusión de kernels.
if USE_COMPILE and hasattr(torch, 'compile'):
    model = torch.compile(model)
    print('torch.compile() aplicado ✓')
else:
    print('torch.compile() no disponible o desactivado')

total_params = sum(p.numel() for p in model.parameters())
print(f'Total params    : {total_params:,}')

## Sección 5 — Infraestructura de entrenamiento

In [ ]:
# Loss con pesos de clase para compensar el desbalance residual
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer con LR diferencial:
#   - backbone conservador (1e-5): preservar features astronómicas de Zoobot
#   - cabeza agresiva (1e-3): adaptar rápidamente a nuestras 6 clases
#   - weight_decay=0.01: mayor regularización que ImageNet models
optimizer = optim.AdamW(
    [
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': head_params,     'lr': LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)

# Scheduler: cosine decay hasta eta_min al final del entrenamiento
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-7
)

# AMP GradScaler (solo activo si hay GPU)
scaler = GradScaler(enabled=USE_AMP)

print('Loss     : CrossEntropyLoss (weighted)')
print('Optim    : AdamW')
print(f'  Backbone LR : {LR_BACKBONE}  (wd={WEIGHT_DECAY})')
print(f'  Head LR     : {LR_HEAD}  (wd={WEIGHT_DECAY})')
print('Schedule : CosineAnnealingLR')
print(f'AMP      : {USE_AMP}')

In [ ]:
# Funciones de checkpoint
def _unwrap_model(m: nn.Module) -> nn.Module:
    """Obtiene el módulo base, ignorando DataParallel y torch.compile()."""
    if isinstance(m, nn.DataParallel):
        m = m.module
    if hasattr(m, '_orig_mod'):   # torch.compile() envuelve en _orig_mod
        m = m._orig_mod
    return m


def save_checkpoint(state: dict, ckpt_dir: pathlib.Path, is_best: bool = False):
    """
    - latest.pth     : sobreescrito en cada epoch (para reanudar)
    - best.pth       : solo cuando mejora el val F1
    - epoch_XXX.pth  : hito cada 5 epochs (archivo permanente)
    """
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save(state, ckpt_dir / 'latest.pth')
    saved = ['latest.pth']
    if is_best:
        torch.save(state, ckpt_dir / 'best.pth')
        saved.append('best.pth')
    if (state['epoch'] + 1) % 5 == 0:
        name = f'epoch_{state["epoch"]+1:03d}.pth'
        torch.save(state, ckpt_dir / name)
        saved.append(name)
    print(f'  [ckpt] saved: {", ".join(saved)}')


def load_checkpoint(
    ckpt_path: pathlib.Path,
    model: nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    scaler: GradScaler,
):
    """
    Restaura modelo, optimizer, scheduler, scaler e historial completo.
    Devuelve (last_epoch, best_val_f1, epochs_no_improve, history).
    """
    ckpt = torch.load(ckpt_path, map_location='cpu')
    _unwrap_model(model).load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    epochs_no_improve = ckpt.get('epochs_no_improve', 0)
    return ckpt['epoch'], ckpt['best_val_f1'], epochs_no_improve, ckpt['history']


print('save_checkpoint / load_checkpoint definidas ✓')

## Sección 6 — Reanudar desde checkpoint

Si existe `latest.pth` en `CKPT_DIR`, el entrenamiento continúa desde el siguiente epoch (incluyendo el contador de early stopping).  
Para empezar desde cero, borra o renombra `latest.pth`.

In [ ]:
LATEST_CKPT = CKPT_DIR / 'latest.pth'

start_epoch       = 0
best_val_f1       = 0.0
epochs_no_improve = 0    # contador para early stopping
history           = []

if LATEST_CKPT.exists():
    print(f'Checkpoint encontrado: {LATEST_CKPT}')
    last_epoch, best_val_f1, epochs_no_improve, history = load_checkpoint(
        LATEST_CKPT, model, optimizer, scheduler, scaler
    )
    start_epoch = last_epoch + 1
    print(f'Reanudando desde epoch {start_epoch + 1}/{EPOCHS}')
    print(f'Mejor val F1         : {best_val_f1:.4f}')
    print(f'Epochs sin mejora    : {epochs_no_improve}/{EARLY_STOP_PAT}')
    print(f'Epochs en historial  : {len(history)}')
else:
    print('Sin checkpoint — entrenamiento desde cero')

print(f'Epochs por entrenar  : {EPOCHS - start_epoch}')

## Sección 7 — Funciones de entrenamiento y validación

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, use_amp):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='  Train', unit='batch', leave=False,
                dynamic_ncols=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast(enabled=use_amp):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        # Gradient clipping para estabilidad en fine-tuning completo
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = running_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1


@torch.no_grad()
def validate(model, loader, criterion, device, use_amp):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='  Val  ', unit='batch', leave=False,
                dynamic_ncols=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(enabled=use_amp):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = running_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1, all_preds, all_labels


print('train_one_epoch / validate definidas ✓')

## Sección 8 — Loop de entrenamiento

In [ ]:
LOG_CSV = LOG_DIR / f'{MODEL_NAME}_log.csv'

print(f'Iniciando entrenamiento: epochs {start_epoch+1} → {EPOCHS}')
print(f'Early stopping: paciencia = {EARLY_STOP_PAT} epochs sin mejora en val F1')
print('=' * 72)

stopped_early = False

for epoch in range(start_epoch, EPOCHS):
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    t0 = time.time()
    ts = datetime.now().strftime('%H:%M:%S')

    print(f'\n[{ts}] ── Epoch {epoch+1:02d}/{EPOCHS} ───────────────────────────────')

    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device, USE_AMP
    )
    val_loss, val_f1, _, _ = validate(
        model, val_loader, criterion, device, USE_AMP
    )
    scheduler.step()

    elapsed = time.time() - t0
    is_best  = val_f1 > best_val_f1

    if is_best:
        best_val_f1       = val_f1
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    row = {
        'epoch':       epoch + 1,
        'train_loss':  round(train_loss, 6),
        'train_f1':    round(train_f1,   6),
        'val_loss':    round(val_loss,   6),
        'val_f1':      round(val_f1,     6),
        'lr_backbone': round(optimizer.param_groups[0]['lr'], 8),
        'lr_head':     round(optimizer.param_groups[1]['lr'], 8),
        'elapsed_s':   round(elapsed, 1),
        'is_best':     is_best,
    }
    history.append(row)

    state = {
        'epoch':                epoch,
        'model_state_dict':     _unwrap_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'best_val_f1':          best_val_f1,
        'epochs_no_improve':    epochs_no_improve,
        'history':              history,
    }
    save_checkpoint(state, CKPT_DIR, is_best=is_best)

    pd.DataFrame(history).to_csv(LOG_CSV, index=False)

    best_tag   = '  ← BEST' if is_best else ''
    early_tag  = f'  [no mejora: {epochs_no_improve}/{EARLY_STOP_PAT}]' if not is_best else ''
    print(
        f'  train  loss={train_loss:.4f}  F1={train_f1:.4f}\n'
        f'  val    loss={val_loss:.4f}  F1={val_f1:.4f}{best_tag}{early_tag}\n'
        f'  time   {elapsed:.0f}s   LR_bb={optimizer.param_groups[0]["lr"]:.2e}',
        flush=True,
    )

    # Early stopping
    if epochs_no_improve >= EARLY_STOP_PAT:
        print(f'\n[Early Stopping] {EARLY_STOP_PAT} epochs sin mejora — deteniendo en epoch {epoch+1}.')
        stopped_early = True
        break

print('\n' + '=' * 72)
if stopped_early:
    print(f'Entrenamiento detenido por early stopping.  Mejor val F1 = {best_val_f1:.4f}')
else:
    print(f'Entrenamiento completo ({EPOCHS} epochs).  Mejor val F1 = {best_val_f1:.4f}')

## Sección 9 — Curvas de entrenamiento

In [ ]:
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} — Training History', fontsize=13, fontweight='bold')

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

best_row = hist_df.loc[hist_df['val_f1'].idxmax()]
axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train')
axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val')
axes[1].axvline(best_row['epoch'], color='red', linestyle='--', alpha=0.5,
                label=f'Best={best_row["val_f1"]:.4f} (ep{int(best_row["epoch"])})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 macro')
axes[1].set_title('Macro F1 Score'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].semilogy(hist_df['epoch'], hist_df['lr_backbone'], label='Backbone')
axes[2].semilogy(hist_df['epoch'], hist_df['lr_head'],     label='Head')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate (log)')
axes[2].set_title('Learning Rate Schedule'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en {LOG_DIR}/{MODEL_NAME}_training_curves.png')

## Sección 10 — Evaluación final (mejor modelo sobre test)

In [ ]:
# Cargar el mejor checkpoint para la evaluación final
best_ckpt = CKPT_DIR / 'best.pth'
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location='cpu')
    _unwrap_model(model).load_state_dict(ckpt['model_state_dict'])
    print(f'Mejor modelo: epoch {ckpt["epoch"]+1}  val F1={ckpt["best_val_f1"]:.4f}')
else:
    print('best.pth no encontrado — usando el estado actual del modelo')

# test_loader creado aquí (no en Sección 3) para no tener workers extra
# vivos durante todo el entrenamiento.
test_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'test.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)
print(f'Test  batches : {len(test_loader):,}  ({len(test_loader.dataset):,} imgs)')

_, _, test_preds, test_labels = validate(model, test_loader, criterion, device, USE_AMP)

print('\nClassification Report — Test set:')
print(classification_report(test_labels, test_preds, target_names=CLASS_ORDER, zero_division=0))

In [ ]:
# Confusion matrix normalizada
cm      = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'{MODEL_NAME} — Confusion Matrix (test set)', fontsize=12, fontweight='bold')

for ax, data, title, fmt in [
    (axes[0], cm,      'Counts',     'd'),
    (axes[1], cm_norm, 'Normalized', '.2f'),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
        ax=ax, vmin=0, vmax=(1 if fmt == '.2f' else None),
    )
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticklabels(CLASS_ORDER, rotation=30, ha='right')
    ax.set_yticklabels(CLASS_ORDER, rotation=0)

plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Sección 11 — Resumen

**Artefactos generados:**
- `models/checkpoints/zoobot_convnext_nano/latest.pth` — último checkpoint
- `models/checkpoints/zoobot_convnext_nano/best.pth` — mejor checkpoint por val F1
- `models/checkpoints/zoobot_convnext_nano/epoch_XXX.pth` — hitos cada 5 epochs
- `logs/zoobot_convnext_nano_log.csv` — historial epoch por epoch
- `logs/zoobot_convnext_nano_training_curves.png`
- `logs/zoobot_convnext_nano_confusion_matrix.png`

**Diferencias respecto a EfficientNet-B3 / ResNet-50 / Swin-S / MaxViT-T:**
- **Preentrenamiento astronómico**: Zoobot fue entrenado con >92M anotaciones de Galaxy Zoo, no solo ImageNet — comprende morfología de galaxias a priori
- **Framework**: usa `FinetuneableZoobotClassifier` de Zoobot, pero con pipeline manual PyTorch (compatible con nuestro sistema de checkpoints)
- **Encoder**: ConvNeXt-Nano (~15M params, más pequeño que todos los otros modelos del pipeline)
- **Fine-tuning completo**: backbone + cabeza, con LR diferencial y layer decay (0.75)
- **LR backbone más bajo** (1e-5 vs 1e-4): las features de Zoobot ya son relevantes para galaxias
- **Mayor paciencia** (10 vs 5): explorar convergencia completa del fine-tuning
- **Mayor regularización** (wd=0.01 vs 1e-4): prevenir overfitting con encoder tan especializado
- **Gradient clipping** (max_norm=1.0): estabilidad en fine-tuning completo
- **Más epochs** (50 vs 30): permitir convergencia total

**Para reanudar el entrenamiento:**
Ejecutar el notebook de nuevo sin borrar `latest.pth`.

**Comparativa acumulada (tras ejecutar 09_evaluation.ipynb):**
- `04` EfficientNet-B3 → best val F1 = 0.6894  /  test F1 = 0.6750  (epoch 16/30)
- `05` ResNet-50       → best val F1 = 0.6914  /  test F1 = 0.6706  (epoch 14/19)
- `06` Swin-S          → best val F1 = 0.6962  /  test F1 = 0.6834  (epoch 25/30)
- `08` MaxViT-T        → best val F1 = 0.6951  /  test F1 = 0.6839  (epoch 17/22)
- `10` **Zoobot ConvNeXt-Nano** → *pendiente de entrenamiento*

**Hipótesis:** Zoobot debería **superar a todos los modelos anteriores** en F1 porque:
1. Su encoder fue preentrenado específicamente en morfología de galaxias (no ImageNet genérico)
2. Ya comprende features como brazos espirales, barras, bordes de galaxia, irregularidades
3. El fine-tuning completo con layer decay adapta todo el modelo preservando el conocimiento astronómico

**Siguiente paso → notebook de evaluación comparativa final**
Cargar todos los `best.pth` y comparar métricas (F1, precision, recall, accuracy) sobre el mismo test set.